# 00 · Explore the LI.FI sample data

Profile of the two provided files, `data/transfers_started.csv` and `data/prices.csv`. Plain pandas, one numbered section per question. Nothing here is a reported metric: numbers quoted in the task documents come from named queries.

Rules applied: both CSVs are loaded with `pandas.read_csv`; `bridge_data` is parsed with `json.loads` so integers stay exact; `minAmount` stays a Python `int` (object column) until it is decimal-adjusted at pricing time; `destinationChainId` stays an integer. No external data: chain names for destination ids and token metadata beyond `prices.csv` are not in the data.

Sections: 1 loading and parsing · 2 rows by chain and day · 3 bridges · 4 integrators and naming variants · 5 tokens and price coverage · 6 destination chain ids · 7 native-asset rows · 8 duplicates · 9 referrer · 10 the two flags · 11 a first USD column · 12 oddity sizes in one table.

In [1]:
import json
from decimal import Decimal
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)

# The notebook runs from analysis/notebooks/; walk up to the repository root that holds data/.
ROOT = Path.cwd().resolve()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data"
TRANSFERS = DATA / "transfers_started.csv"
PRICES = DATA / "prices.csv"
for f in (TRANSFERS, PRICES):
    if not f.exists():
        raise FileNotFoundError(f"{f} is missing: drop the provided CSV into data/ (see analysis/README.md)")

ZERO_ADDRESS = "0x0000000000000000000000000000000000000000"
EXPORT_START = pd.Timestamp("2026-10-13 00:00:00", tz="UTC")   # inclusive, data/README_data_final.md
EXPORT_END = pd.Timestamp("2026-10-20 00:00:00", tz="UTC")     # exclusive
print("data dir:", DATA.relative_to(ROOT))

data dir: data


## 1. Loading and parsing

`read_csv` with default types gives strings for the hex columns and int64 for `block_number` and `log_index`. `bridge_data` is a JSON string per row; `json.loads` turns it into a dict whose integers are exact Python ints, which matters because `minAmount` exceeds int64 (checked below). The parsed keys become columns next to the event columns in one frame, `tf`.

In [2]:
raw = pd.read_csv(TRANSFERS)
prices = pd.read_csv(PRICES)
print("transfers_started.csv:", raw.shape, "  prices.csv:", prices.shape)
raw.dtypes

transfers_started.csv: (124858, 8)   prices.csv: (40754, 6)


chain                 str
block_time            str
block_number        int64
tx_hash               str
log_index           int64
contract_address      str
tx_from               str
bridge_data           str
dtype: object

In [3]:
bridge = pd.DataFrame([json.loads(s) for s in raw["bridge_data"]])
tf = pd.concat([raw.drop(columns="bridge_data"), bridge], axis=1)
tf["block_time"] = pd.to_datetime(tf["block_time"], utc=True)
tf["hour"] = tf["block_time"].dt.floor("h")          # the join key to prices.hour (start of hour)
prices["hour"] = pd.to_datetime(prices["hour"], utc=True)
tf.dtypes

chain                                 str
block_time            datetime64[us, UTC]
block_number                        int64
tx_hash                               str
log_index                           int64
contract_address                      str
tx_from                               str
transactionId                         str
bridge                                str
integrator                            str
referrer                              str
sendingAssetId                        str
receiver                              str
minAmount                          object
destinationChainId                  int64
hasSourceSwaps                       bool
hasDestinationCall                   bool
hour                  datetime64[us, UTC]
dtype: object

In [4]:
# Every bridge_data object carries the same ten keys; no nulls in either file.
key_sets = pd.Series([tuple(sorted(json.loads(s).keys())) for s in raw["bridge_data"]]).value_counts()
print("distinct key sets in bridge_data:", len(key_sets))
print(key_sets.index[0])
print("nulls, transfers:", int(tf.isna().sum().sum()), "  nulls, prices:", int(prices.isna().sum().sum()))

distinct key sets in bridge_data: 1
('bridge', 'destinationChainId', 'hasDestinationCall', 'hasSourceSwaps', 'integrator', 'minAmount', 'receiver', 'referrer', 'sendingAssetId', 'transactionId')
nulls, transfers: 0   nulls, prices: 0


In [5]:
# minAmount: exact Python ints; the largest exceeds int64, so the column must stay object, never float64 or int64.
print("minAmount python types:", tf["minAmount"].map(type).value_counts().to_dict())
print("max minAmount:", max(tf["minAmount"]), " > int64 max:", max(tf["minAmount"]) > 2**63 - 1)
print("minAmount == 0 rows:", int((tf["minAmount"] == 0).sum()))
print("destinationChainId dtype:", tf["destinationChainId"].dtype, " max:", tf["destinationChainId"].max())
print("flags dtype:", tf["hasSourceSwaps"].dtype, tf["hasDestinationCall"].dtype)

minAmount python types: {<class 'int'>: 124858}
max minAmount: 20273815112416660413525450  > int64 max: True
minAmount == 0 rows: 0
destinationChainId dtype: int64  max: 9270000000000000
flags dtype: bool bool


In [6]:
# Event-level columns: one contract address on all six chains; addresses lowercase 42-char hex; file sorted as the README says.
print("contract_address values:", tf["contract_address"].unique().tolist())
for col in ["tx_hash", "contract_address", "tx_from", "sendingAssetId", "receiver", "referrer"]:
    print(f"{col:18s} lengths {tf[col].str.len().unique().tolist()}  uppercase hex chars: {int(tf[col].str.contains('[A-F]').sum())}")
print("transactionId length:", tf["transactionId"].str.len().unique().tolist())
in_order = (tf.sort_values(["block_time", "chain", "log_index"], kind="stable").index == tf.index).all()
print("file ordered by (block_time, chain, log_index):", bool(in_order))
tf.head(3).T

contract_address values: ['0x1231deb6f5749ef6ce6943a275a1d3e7486f4eae']
tx_hash            lengths [66]  uppercase hex chars: 0
contract_address   lengths [42]  uppercase hex chars: 0


tx_from            lengths [42]  uppercase hex chars: 0


sendingAssetId     lengths [42]  uppercase hex chars: 0


receiver           lengths [42]  uppercase hex chars: 0
referrer           lengths [42]  uppercase hex chars: 0
transactionId length: [66]
file ordered by (block_time, chain, log_index): True


,0,1,2
chain,polygon,bnb,arbitrum
block_time,2026-10-13 00:00:00+00:00,2026-10-13 00:00:01+00:00,2026-10-13 00:00:05+00:00
block_number,95429540,127303032,514268491
tx_hash,0xcb12b8f93290964d5aa86cc282c934cc7480ed371f30c9ec5e8cf07da4b3fd69,0x5a2d3f65316f28dc0d1a10125a28f8de2a19546441333ecbac6ef6fb6cae12ac,0xe7254c0d62dbc26947a6d567289cdf00e0910e94897d8b9f72c4fc2d9dd96153
log_index,1285,878,28
contract_address,0x1231deb6f5749ef6ce6943a275a1d3e7486f4eae,0x1231deb6f5749ef6ce6943a275a1d3e7486f4eae,0x1231deb6f5749ef6ce6943a275a1d3e7486f4eae
tx_from,0x4337041d8047569945950ec533e3f1af96d7be70,0x29d1630ce957e4663b6c031187cde26aa77f1dc1,0x8d232f52a9424a662d410feba459cc7de44de2bb
transactionId,0x7bf59f576d4b99dccadc95e715b2da0dc01e59d6104762af7a7f7df926f81a1e,0x4f5688d431cd3b04d94a2578fb95e3e61aecb27ce38ef964c927c8849d3ea121,0xa65297c0b493701691df1791a42e0c0480fcb19406f6bd94e081c07417921c34
bridge,relaydepository,across,mayan
integrator,_chainless,_binancewallet,rubic


## 2. Row counts by chain and by day, and the export window check

The README promises six source chains and seven full UTC days, 2026-10-13 00:00 inclusive to 2026-10-20 00:00 exclusive.

In [7]:
print("min block_time:", tf["block_time"].min(), "  max block_time:", tf["block_time"].max())
inside = tf["block_time"].between(EXPORT_START, EXPORT_END, inclusive="left")
print("rows inside the export window:", int(inside.sum()), "of", len(tf), " outside:", int((~inside).sum()))
print("distinct hours with at least one row:", tf["hour"].nunique(), "of 168")
tf["chain"].value_counts().to_frame("rows").assign(share=lambda d: (d["rows"] / len(tf)).round(4))

min block_time: 2026-10-13 00:00:00+00:00   max block_time: 2026-10-19 23:59:49+00:00
rows inside the export window: 124858 of 124858  outside: 0
distinct hours with at least one row: 168 of 168


,rows,share
chain,,
base,34908,0.2796
ethereum,29351,0.2351
bnb,27052,0.2167
arbitrum,20398,0.1634
polygon,7393,0.0592
optimism,5756,0.0461


In [8]:
by_day = tf.groupby([tf["block_time"].dt.date.rename("day"), "chain"]).size().unstack("chain")
by_day["total"] = by_day.sum(axis=1)
by_day.loc["total"] = by_day.sum()
by_day

chain,arbitrum,base,bnb,ethereum,optimism,polygon,total
day,,,,,,,
2026-10-13,3077,5414,4320,4524,852,1223,19410
2026-10-14,2923,5458,4275,4328,848,1160,18992
2026-10-15,3302,5619,4337,4640,869,1178,19945
2026-10-16,2763,5130,3748,4185,983,993,17802
2026-10-17,2466,4388,3439,3670,787,924,15674
2026-10-18,2586,4425,3279,3683,623,864,15460
2026-10-19,3281,4474,3654,4321,794,1051,17575
total,20398,34908,27052,29351,5756,7393,124858


## 3. Bridges

`bridge` is the tool name the LI.FI contract was called with. Names are case-sensitive strings; several share a stem (`mayan`, `mayanMCTP`, `mayanFastMCTP`, `mayanWH`; `polymer`, `polymerStandard`; `lifiIntents`, `lifiIntentsDex`; `celercircle`, `celercirclefast`; `m0`, `m0Dex`; `paxos`, `paxosDex`), and some bridge names are chain names (`arbitrum`, `polygon`, `gnosis`, `megaeth`, `superset`), presumably the canonical bridge of that chain.

In [9]:
bridges = tf["bridge"].value_counts().to_frame("rows")
bridges["share"] = (bridges["rows"] / len(tf)).round(4)
print("distinct bridges:", len(bridges), "  distinct after lower-casing:", tf["bridge"].str.lower().nunique())
bridges

distinct bridges: 31   distinct after lower-casing: 31


,rows,share
bridge,,
across,34163,0.2736
relaydepository,31483,0.2522
lifiIntents,16185,0.1296
layerswap,12659,0.1014
mayan,11422,0.0915
polymerStandard,6631,0.0531
gasZipBridge,3678,0.0295
near,2814,0.0225
polymer,1320,0.0106


In [10]:
# Bridges by source chain: which tools run where.
pd.crosstab(tf["bridge"], tf["chain"]).sort_values("ethereum", ascending=False)

chain,arbitrum,base,bnb,ethereum,optimism,polygon
bridge,,,,,,
across,3840,5918,10543,9930,880,3052
relaydepository,4137,7715,9384,7857,1120,1270
mayan,1936,4018,1908,2439,395,726
layerswap,2600,5324,572,2098,1384,681
lifiIntents,4344,5240,3626,1731,782,462
polymerStandard,1253,2810,0,1446,367,755
near,745,750,70,1140,33,76
gasZipBridge,620,1217,474,939,428,0
symbiosis,121,85,200,318,6,42


## 4. Integrators, counts and naming variants

`integrator` is a free string passed by the caller. This section counts raw values, then looks for variants that are plausibly the same integrator: case-only differences, a leading underscore, environment suffixes (`-prod`, `-dev`, `-test`, `-staging`, `-api`, …), and names that are a prefix of another name. Which of these to merge is a definition question, treated in `task_1.md`.

In [11]:
integrators = tf["integrator"].value_counts()
print("distinct raw integrator strings:", len(integrators))
print("empty-string integrator rows:", int((tf["integrator"] == "").sum()))
print("rows whose integrator has leading/trailing whitespace:", int(tf["integrator"].str.strip().ne(tf["integrator"]).sum()))
print("integrators with exactly 1 row:", int((integrators == 1).sum()), "  with < 10 rows:", int((integrators < 10).sum()), "  with >= 100 rows:", int((integrators >= 100).sum()))
share = integrators.cumsum() / len(tf)
print("row share of top 1 / 5 / 10 / 20 integrators:", [round(float(share.iloc[k - 1]), 4) for k in (1, 5, 10, 20)])
print("the empty-string rows:")
display(tf[tf["integrator"] == ""][["chain", "block_time", "bridge", "sendingAssetId", "minAmount", "destinationChainId"]])
integrators.head(40).to_frame("rows")

distinct raw integrator strings: 598
empty-string integrator rows: 2


rows whose integrator has leading/trailing whitespace: 0
integrators with exactly 1 row: 146   with < 10 rows: 391   with >= 100 rows: 66
row share of top 1 / 5 / 10 / 20 integrators: [0.3276, 0.6164, 0.7453, 0.8477]
the empty-string rows:


,chain,block_time,bridge,sendingAssetId,minAmount,destinationChainId
5389,polygon,2026-10-13 07:37:23+00:00,across,0x3c499c542cef5e3811e1192ce70d8cc03d5c3359,1062393,42161
6020,polygon,2026-10-13 08:21:23+00:00,across,0x3c499c542cef5e3811e1192ce70d8cc03d5c3359,2026909,42161


,rows
integrator,
jumper.exchange,40908
_binancewallet,15109
rabbykmsv2,7759
metamask-bridge,7332
base-app,5859
phantom,5233
lifi-api,3465
tangem,2621
bitget,2534


In [12]:
# Case-only variants: none. Punctuation-only variants: none.
lower_groups = tf.groupby(tf["integrator"].str.lower())["integrator"].agg(lambda s: sorted(set(s)))
print("names that differ only by case:", lower_groups[lower_groups.map(len) > 1].to_dict())
alnum = tf["integrator"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
punct_groups = tf.groupby(alnum)["integrator"].agg(lambda s: sorted(set(s)))
print("names that differ only by punctuation:", punct_groups[punct_groups.map(len) > 1].to_dict())
print("names containing uppercase letters:", int((integrators.index != integrators.index.str.lower()).sum()))

names that differ only by case: {}


names that differ only by punctuation: {}
names containing uppercase letters: 108


In [13]:
# Leading underscore: six names, a naming convention the data does not explain.
underscore = tf[tf["integrator"].str.startswith("_")]["integrator"].value_counts()
print("rows with a leading-underscore integrator:", int(underscore.sum()))
underscore.to_frame("rows")

rows with a leading-underscore integrator: 19068


,rows
integrator,
_binancewallet,15109
_arbitrum,1688
_ledger,1501
_chainless,551
_bitso,213
_limited,6


In [14]:
# Environment-like suffixes and tokens inside names (substring match, case-insensitive): candidates for one integrator reported under several strings.
patterns = ["prod", "production", "dev", "test", "staging", "stg", "local", "api", "sdk", "widget", "v1", "v2", "v3"]
rows = []
for pat in patterns:
    m = integrators.index.str.lower().str.contains(pat, regex=False)
    rows.append({"pattern": pat, "names": int(m.sum()), "rows": int(integrators[m].sum()), "examples": ", ".join(sorted(integrators.index[m])[:6])})
pd.DataFrame(rows)

,pattern,names,rows,examples
0,prod,24,3643,"Banxa-EUIV---Production, Banxa-GIV---Prod-API, Byrrgis-Production, Dispence-..."
1,production,5,242,"Banxa-EUIV---Production, Byrrgis-Production, Quicknode---Production, coin98-..."
2,dev,6,130,"HashPort-Wallet-Dev, adev, dev.jumper.exchange, lifi-develop-api, swapiodev,..."
3,test,7,273,"ab-dex-test, base-app-testing, bcdc_testing, georgi-test, gmgntestenv, heyAu..."
4,staging,2,69,"Curvy-Staging, trillions-staging"
5,stg,3,13,"Reah-STG, jumper.exchange.earnstg, stg.jumper.exchange"
6,local,4,117,"BridgerLocal, Tw-Local-Routing, lifi-local-api, netzwert-local"
7,api,12,4247,"Banxa-GIV---Prod-API, LIFI-BUMI-PROD-API, coinapi-v2, lifi-api, lifi-develop..."
8,sdk,2,9,"lifi-sdk, walletconnect-agent-sdk"
9,widget,1,134,perps-widget


In [15]:
# Families: strip a leading underscore and repeated environment suffixes, then group. Only families with more than one raw name are shown.
import re
SUFFIX = r"(?:-{1,3}|_|\.)?(?:production|prod|dev|develop|development|testing|testenv|test|staging|stg|local|api|sdk|widget|app|v\d+)$"
def stem(name):
    s = name.lower().lstrip("_")
    prev = None
    while prev != s:
        prev, s = s, re.sub(SUFFIX, "", s)
    return re.sub(r"[^a-z0-9]", "", s)
tf["integrator_stem"] = tf["integrator"].map(stem)
families = tf.groupby("integrator_stem")["integrator"].agg(lambda s: sorted(set(s)))
families = families[families.map(len) > 1]
fam_rows = tf[tf["integrator_stem"].isin(families.index)]
print("families with more than one raw name:", len(families), "  rows in them:", len(fam_rows))
pd.DataFrame({"raw names": families.map(lambda v: [(n, int(integrators[n])) for n in v])})

families with more than one raw name: 7   rows in them: 11252


,raw names
integrator_stem,
base,"[(base-app, 5859), (base-app-testing, 2)]"
bcdc,"[(bcdc, 240), (bcdc_testing, 7)]"
hashportwallet,"[(HashPort-Wallet-Dev, 7), (HashPort-Wallet-Prod, 827)]"
heyaura,"[(heyAura, 177), (heyAura-test, 1)]"
jumperexchangeearn,"[(jumper.exchange.earn, 15), (jumper.exchange.earnstg, 1)]"
lifi,"[(lifi-api, 3465), (lifi-develop-api, 66), (lifi-local-api, 5), (lifi-sdk, 7..."
tria,"[(tria, 529), (tria-dev, 41)]"


In [16]:
# Names that are a strict prefix of another name (case-insensitive), with row counts: some are the same product (jumper.exchange / jumper.exchange.earn), some are not (hey / heyelsa).
names = sorted(set(tf["integrator"]) - {""})
prefix_pairs = [(a, b, int(integrators[a]), int(integrators[b])) for a in names for b in names if a != b and b.lower().startswith(a.lower())]
pd.DataFrame(prefix_pairs, columns=["name", "longer name", "rows", "rows of longer"])

,name,longer name,rows,rows of longer
0,Vault,vaultnow,1,1
1,Vault,vaulto-stake,1,1
2,base-app,base-app-testing,5859,2
3,bcdc,bcdc_testing,240,7
4,blazpay,blazpay-defi,3,1
5,dcent,dcent_region,67,21
6,flow,flowfi,6,3
7,hermes,hermes-ces,5,1
8,hermes,hermes-ct,5,2
9,hey,heyAura,9,177


In [17]:
# Stems shared with LI.FI's own tooling and with Jumper (LI.FI's own front end), to decide what counts as a third-party integrator.
print("names containing 'lifi' or 'li.fi':", [(n, int(integrators[n])) for n in names if re.search(r"li\.?fi", n.lower())])
print("names containing 'jumper':", [(n, int(integrators[n])) for n in names if "jumper" in n.lower()])
tf.groupby("chain")["integrator"].nunique().to_frame("distinct integrators")

names containing 'lifi' or 'li.fi': [('LIFI-BUMI-PROD-API', 56), ('kaitolifiswap', 9), ('li.fi-playground', 174), ('li.fi-solver', 92), ('lifi-api', 3465), ('lifi-cli', 1), ('lifi-develop-api', 66), ('lifi-local-api', 5), ('lifi-sc-tools', 1), ('lifi-sdk', 7), ('lifitestapp', 3), ('ronin.wallet.lifi', 653)]
names containing 'jumper': [('dev.jumper.exchange', 2), ('jumper.exchange', 40908), ('jumper.exchange.earn', 15), ('jumper.exchange.earnstg', 1), ('jumperadvanced', 1119), ('jumperrwa', 30), ('stg.jumper.exchange', 2)]


,distinct integrators
chain,
arbitrum,241
base,325
bnb,259
ethereum,320
optimism,96
polygon,163


## 5. Tokens per chain, the prices file, and price coverage per hour

`sendingAssetId` is the token contract address on the source chain, with the zero address for the native asset. `prices.csv` is keyed by `(chain, token_address, hour)` and carries `symbol` and `decimals`; it is the only source of decimals in the data, so a token without a price row cannot even be converted to whole-token units.

In [18]:
print("prices rows:", len(prices), "  distinct (chain, token):", prices.groupby(["chain", "token_address"]).ngroups,
      "  distinct hours:", prices["hour"].nunique(), "  hour range:", prices["hour"].min(), "to", prices["hour"].max())
print("duplicate (chain, token, hour) keys:", int(prices.duplicated(["chain", "token_address", "hour"]).sum()))
consistency = prices.groupby(["chain", "token_address"])[["symbol", "decimals"]].nunique()
print("tokens with more than one symbol:", int((consistency["symbol"] > 1).sum()), "  more than one decimals:", int((consistency["decimals"] > 1).sum()))
print("price_usd <= 0:", int((prices["price_usd"] <= 0).sum()), "  price_usd range:", prices["price_usd"].min(), "to", prices["price_usd"].max())
hours_per_token = prices.groupby(["chain", "token_address"])["hour"].nunique()
print("tokens with all 168 hours:", int((hours_per_token == 168).sum()), " of", len(hours_per_token))
partial = prices.groupby(["chain", "token_address", "symbol"])["hour"].agg(["nunique", "min", "max"])
partial[partial["nunique"] < 168]

prices rows: 40754   distinct (chain, token): 244   distinct hours: 168   hour range: 2026-10-13 00:00:00+00:00 to 2026-10-19 23:00:00+00:00
duplicate (chain, token, hour) keys: 0
tokens with more than one symbol: 0   more than one decimals: 0
price_usd <= 0: 0   price_usd range: 3e-08 to 91074.68232328685
tokens with all 168 hours: 240  of 244


nunique                       min                       max
chain    token_address                              symbol                                                             
arbitrum 0x27f6c8289550fce67f6b50bed1f519966afe5287 tGBP        152 2026-10-13 16:00:00+00:00 2026-10-19 23:00:00+00:00
ethereum 0x280839980a7ed0d7717f64125fe241012e5f5815 strUSD       33 2026-10-13 00:00:00+00:00 2026-10-14 08:00:00+00:00
         0x437cc33344a0b27a429f795ff6b469c72698b291 wM          145 2026-10-13 00:00:00+00:00 2026-10-19 23:00:00+00:00
         0xb0f70c0bd6fd87dbeb7c10dc692a2a6106817072 BTC.b       104 2026-10-15 16:00:00+00:00 2026-10-19 23:00:00+00:00

In [19]:
price_tokens = prices[["chain", "token_address", "symbol", "decimals"]].drop_duplicates()
print("distinct symbols:", price_tokens["symbol"].nunique())
print("decimals distribution over (chain, token):", price_tokens["decimals"].value_counts().to_dict())
print("symbols with more than one address on the same chain:")
dup_sym = price_tokens.groupby(["chain", "symbol"]).size()
display(price_tokens.set_index(["chain", "symbol"]).loc[dup_sym[dup_sym > 1].index].sort_index())
print("symbols whose decimals differ across chains:")
dec = price_tokens.groupby("symbol")["decimals"].nunique()
display(price_tokens[price_tokens["symbol"].isin(dec[dec > 1].index)].sort_values(["symbol", "chain"]))
print("tokens with unusual decimals (not 6, 8, 18):")
display(price_tokens[~price_tokens["decimals"].isin([6, 8, 18])])

distinct symbols: 155
decimals distribution over (chain, token): {18: 192, 6: 34, 8: 15, 9: 2, 2: 1}
symbols with more than one address on the same chain:


token_address  decimals
chain    symbol                                                      
arbitrum USDC    0xaf88d065e77c8cc2239327c5edb3a432268e5831         6
         USDC    0xff970a61a04b1ca14834a43f5de4533ebddb5cc8         6
optimism USDC    0x0b2c639c533813f4aa9d7837caf62653d097ff85         6
         USDC    0x7f5c764cbc14f9669b88837ca1490cca17c31607         6
polygon  USDC    0x2791bca1f2de4661ed88a30c99a7a9449aa84174         6
         USDC    0x3c499c542cef5e3811e1192ce70d8cc03d5c3359         6

symbols whose decimals differ across chains:


,chain,token_address,symbol,decimals
3512,arbitrum,0xaf88d065e77c8cc2239327c5edb3a432268e5831,USDC,6
5360,arbitrum,0xff970a61a04b1ca14834a43f5de4533ebddb5cc8,USDC,6
9728,base,0x833589fcd6edb6e08f4c7c32d4f71b54bda02913,USDC,6
16952,bnb,0x8ac76a51cc950d9822d68b83fe1ad97b32cd580d,USDC,18
27882,ethereum,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,6
35042,optimism,0x0b2c639c533813f4aa9d7837caf62653d097ff85,USDC,6
36386,optimism,0x7f5c764cbc14f9669b88837ca1490cca17c31607,USDC,6
37898,polygon,0x2791bca1f2de4661ed88a30c99a7a9449aa84174,USDC,6
38066,polygon,0x3c499c542cef5e3811e1192ce70d8cc03d5c3359,USDC,6
14264,base,0xfde4c96c8593536e31f229ea8f37b2ada2699bb2,USDT,6


tokens with unusual decimals (not 6, 8, 18):


,chain,token_address,symbol,decimals
22338,ethereum,0x582d872a1b094fc48f5de31d3b73f2d9be47def1,TONCOIN,9
32186,ethereum,0xd31a59c85ae9d8edefec411d448f90841571b89c,SOL,9
40250,polygon,0xdf7837de1f2fa4631d716cf2502f8b230f1dcc32,TEL,2


In [20]:
# Tokens that appear in transfers, per chain, with their price-file metadata.
tokens = tf.groupby(["chain", "sendingAssetId"]).size().rename("rows").reset_index()
tokens = tokens.merge(price_tokens, left_on=["chain", "sendingAssetId"], right_on=["chain", "token_address"], how="left").drop(columns="token_address")
tokens["priced_any_hour"] = tokens["symbol"].notna()
print("distinct (chain, token) in transfers:", len(tokens), "  distinct addresses across chains:", tf["sendingAssetId"].nunique())
print("price-file tokens that never appear in transfers:", int((~price_tokens.set_index(["chain", "token_address"]).index.isin(tokens.set_index(["chain", "sendingAssetId"]).index)).sum()))
print("(chain, token) with no price row at all:", int((~tokens["priced_any_hour"]).sum()), " covering rows:", int(tokens.loc[~tokens["priced_any_hour"], "rows"].sum()))
display(tf.groupby("chain")["sendingAssetId"].nunique().to_frame("distinct tokens"))
tokens.sort_values("rows", ascending=False).head(30)

distinct (chain, token) in transfers: 255   distinct addresses across chains: 228
price-file tokens that never appear in transfers: 0
(chain, token) with no price row at all: 11  covering rows: 380


,distinct tokens
chain,
arbitrum,35
base,55
bnb,26
ethereum,100
optimism,17
polygon,22


,chain,sendingAssetId,rows,symbol,decimals,priced_any_hour
61,base,0x833589fcd6edb6e08f4c7c32d4f71b54bda02913,17758,USDC,6.0,True
35,base,0x0000000000000000000000000000000000000000,13519,ETH,18.0,True
116,ethereum,0x0000000000000000000000000000000000000000,12797,ETH,18.0,True
99,bnb,0x55d398326f99059ff775485246999027b3197955,11884,USDT,18.0,True
23,arbitrum,0xaf88d065e77c8cc2239327c5edb3a432268e5831,10638,USDC,6.0,True
175,ethereum,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,8944,USDC,6.0,True
0,arbitrum,0x0000000000000000000000000000000000000000,6651,ETH,18.0,True
94,bnb,0x2170ed0880ac9a755fd29b2688956bd959f933f8,5433,ETH,18.0,True
106,bnb,0x8ac76a51cc950d9822d68b83fe1ad97b32cd580d,5308,USDC,18.0,True
204,ethereum,0xdac17f958d2ee523a2206206994597c13d831ec7,4453,USDT,6.0,True


In [21]:
# Never-priced (chain, token) pairs: polygon's native asset is the only one with material volume.
tokens[~tokens["priced_any_hour"]].sort_values("rows", ascending=False)

,chain,sendingAssetId,rows,symbol,decimals,priced_any_hour
233,polygon,0x0000000000000000000000000000000000000000,364,NaN,NaN,False
19,arbitrum,0x80eede496655fb9047dd39d9f418d5483ed600df,3,NaN,NaN,False
222,optimism,0x43edd7f3831b08fe70b7555ddd373c8bf65a9050,3,NaN,NaN,False
91,bnb,0x02d608506ca0048d0d991a11f1e7fb8cad1e44f8,2,NaN,NaN,False
244,polygon,0x80eede496655fb9047dd39d9f418d5483ed600df,2,NaN,NaN,False
2,arbitrum,0x03569cc076654f82679c4ba2124d64774781b01d,1,NaN,NaN,False
60,base,0x7094c27f342dbadfbbed005b219431595e33b305,1,NaN,NaN,False
80,base,0xd6122ddada244913521f3d62006eaf756c157660,1,NaN,NaN,False
108,bnb,0x91f9cc2649ac70a071602cade9b0c1a5868af51d,1,NaN,NaN,False
114,bnb,0xec93fe7ff4b09ca3ccafbc4cc9615e62be412780,1,NaN,NaN,False


In [22]:
# Price coverage per hour: left-join each transfer to the price of its (chain, token, hour). The row count must not change.
joined = tf.merge(prices[["chain", "token_address", "hour", "symbol", "decimals", "price_usd"]],
                  left_on=["chain", "sendingAssetId", "hour"], right_on=["chain", "token_address", "hour"], how="left").drop(columns="token_address")
assert len(joined) == len(tf)
joined["price_missing"] = joined["price_usd"].isna()
joined["token_priced_any_hour"] = joined.set_index(["chain", "sendingAssetId"]).index.isin(price_tokens.set_index(["chain", "token_address"]).index)
n_missing = int(joined["price_missing"].sum())
print("rows with no price for their (chain, token, hour):", n_missing, f"({n_missing / len(tf):.4%})")
print("  token never priced:", int((joined["price_missing"] & ~joined["token_priced_any_hour"]).sum()),
      "  token priced in other hours but not this one:", int((joined["price_missing"] & joined["token_priced_any_hour"]).sum()))
missing_hour = joined[joined["price_missing"] & joined["token_priced_any_hour"]]
display(missing_hour[["chain", "block_time", "hour", "sendingAssetId", "bridge", "integrator", "minAmount"]])
print("their tokens in prices.csv:")
display(price_tokens.merge(missing_hour[["chain", "sendingAssetId"]].drop_duplicates(), left_on=["chain", "token_address"], right_on=["chain", "sendingAssetId"]).drop(columns="sendingAssetId"))
joined.groupby("chain")["price_missing"].agg(rows="size", missing="sum", share="mean").assign(share=lambda d: d["share"].round(4))

rows with no price for their (chain, token, hour): 381 (0.3051%)
  token never priced: 380   token priced in other hours but not this one: 1


,chain,block_time,hour,sendingAssetId,bridge,integrator,minAmount
120246,ethereum,2026-10-19 17:53:47+00:00,2026-10-19 17:00:00+00:00,0x280839980a7ed0d7717f64125fe241012e5f5815,glacis,jumper.exchange,95359256478038709086


their tokens in prices.csv:


,chain,token_address,symbol,decimals
0,ethereum,0x280839980a7ed0d7717f64125fe241012e5f5815,strUSD,18


,rows,missing,share
chain,,,
arbitrum,20398,4,0.0002
base,34908,2,0.0001
bnb,27052,4,0.0001
ethereum,29351,2,0.0001
optimism,5756,3,0.0005
polygon,7393,366,0.0495


In [23]:
# Price-side observations that affect any USD figure: cross-chain price identity for major symbols, stablecoin deviation from 1, hourly jumps on thin tokens.
by_symbol_hour = prices.pivot_table(index=["symbol", "hour"], columns="chain", values="price_usd")
spread = {}
for s in ["ETH", "WETH", "USDC", "USDT", "WBTC"]:
    sub = by_symbol_hour.loc[s].dropna(axis=1, how="all")
    rel = (sub.max(axis=1) - sub.min(axis=1)) / sub.min(axis=1)
    spread[s] = {"chains": len(sub.columns), "max relative spread across chains in one hour": round(float(rel.max()), 6)}
display(pd.DataFrame(spread).T)
stables = prices[prices["symbol"].isin(["USDC", "USDT", "USDT0", "USD₮0", "DAI", "USDe", "PYUSD", "USDG", "USD1"])]
display(stables.groupby("symbol")["price_usd"].agg(["min", "max"]))
p_sorted = prices.sort_values(["chain", "token_address", "hour"])
p_sorted["abs_hourly_return"] = p_sorted.groupby(["chain", "token_address"])["price_usd"].pct_change().abs()
print("share of hourly price moves above 5%:", round(float((p_sorted["abs_hourly_return"] > 0.05).mean()), 4))
print("ETH on ethereum, min/max over the week:", prices.query("chain == 'ethereum' and symbol == 'ETH'")["price_usd"].agg(["min", "max"]).round(2).to_dict())
p_sorted.groupby(["chain", "symbol"])["abs_hourly_return"].max().sort_values(ascending=False).head(10).to_frame("max abs hourly return")

,chains,max relative spread across chains in one hour
ETH,5.0,0.000000
WETH,5.0,0.000347
USDC,6.0,0.000000
USDT,4.0,0.000000
WBTC,5.0,0.006682


,min,max
symbol,,
DAI,0.999840,1.000266
PYUSD,0.994362,1.001912
USD1,0.998513,0.999358
USDC,1.000018,1.000317
USDG,0.999800,1.001073
USDT,0.998687,0.999402
USDT0,0.998687,0.999402
USDe,0.999473,1.000624
USD₮0,0.996443,1.000217


share of hourly price moves above 5%: 0.0088


ETH on ethereum, min/max over the week: {'min': 2341.95, 'max': 2529.93}


,,max abs hourly return
chain,symbol,
bnb,PUFFER,1.627738
ethereum,GEAR,1.550763
optimism,kBTC,0.904418
base,ISLAND,0.566836
bnb,AGT,0.316502
ethereum,SAHARA,0.300699
bnb,GAME,0.263737
polygon,COCA,0.229278
ethereum,ZRC,0.228829


## 6. Destination chain ids

`destinationChainId` is an integer; 69 distinct values. The data does not name them. Internal evidence exists for three: the bridges named `arbitrum`, `polygon` and `gnosis` run from ethereum to 42161, 137 and 100 only. "Outside the EVM range" has no definition in the data, so three candidate rules are sized: id above 2^32−1, id above 2^53−1, and receiver equal to the sentinel `0x11f1…f1` that appears whenever the destination cannot hold a 20-byte address.

In [24]:
dest = tf["destinationChainId"].value_counts().to_frame("rows")
dest["share"] = (dest["rows"] / len(tf)).round(4)
dest["above_2^32"] = dest.index > 2**32 - 1
dest["above_2^53"] = dest.index > 2**53 - 1
print("distinct destination ids:", len(dest), "  ids <= 0:", int((tf["destinationChainId"] <= 0).sum()))
dest

distinct destination ids: 69   ids <= 0: 0


,rows,share,above_2^32,above_2^53
destinationChainId,,,,
4663,27036,0.2165,False,False
1,18893,0.1513,False,False
8453,14886,0.1192,False,False
1151111081099710,14229,0.1140,True,False
56,13334,0.1068,False,False
42161,11522,0.0923,False,False
137,5945,0.0476,False,False
10,2888,0.0231,False,False
999,2268,0.0182,False,False


In [25]:
SENTINEL = "0x11f111f111f111f111f111f111f111f111f111f1"
above32 = tf["destinationChainId"] > 2**32 - 1
above53 = tf["destinationChainId"] > 2**53 - 1
sentinel = tf["receiver"] == SENTINEL
print("rows with id > 2^32-1:", int(above32.sum()), f"({above32.mean():.4%})", " distinct ids:", tf.loc[above32, "destinationChainId"].nunique())
print("rows with id > 2^53-1:", int(above53.sum()), " distinct ids:", tf.loc[above53, "destinationChainId"].nunique())
print("rows with receiver == sentinel:", int(sentinel.sum()), " distinct ids:", tf.loc[sentinel, "destinationChainId"].nunique())
display(pd.crosstab(above32.rename("id > 2^32-1"), sentinel.rename("receiver is sentinel")))
print("sentinel receiver with an id below 2^32:")
display(tf[sentinel & ~above32][["chain", "bridge", "integrator", "destinationChainId", "receiver"]])
pd.crosstab(tf.loc[above32, "destinationChainId"], tf.loc[above32, "bridge"])

rows with id > 2^32-1: 16666 (13.3480%)  distinct ids: 6
rows with id > 2^53-1: 167  distinct ids: 1
rows with receiver == sentinel: 16668  distinct ids: 7


receiver is sentinel,False,True
id > 2^32-1,,
False,108190,2
True,0,16666


sentinel receiver with an id below 2^32:


,chain,bridge,integrator,destinationChainId,receiver
116776,arbitrum,lifiIntents,lifi-local-api,728126428,0x11f111f111f111f111f111f111f111f111f111f1
116822,polygon,lifiIntents,lifi-local-api,728126428,0x11f111f111f111f111f111f111f111f111f111f1


bridge,chainflip,glacis,layerswap,lifiIntents,mayan,mayanFastMCTP,mayanMCTP,mayanWH,near,polymer,polymerStandard,relaydepository,symbiosis
destinationChainId,,,,,,,,,,,,,
20000000000001,12,0,994,0,0,0,0,0,84,0,0,478,0
20000000000005,0,0,0,0,0,0,0,0,13,0,0,0,0
1151111081099710,46,8,120,371,5932,49,36,1,1547,91,580,5448,0
1201081091099710,0,0,0,0,0,0,0,0,1,3,0,0,0
1885080386571452,0,0,76,0,0,0,0,0,513,0,0,0,96
9270000000000000,0,0,0,0,0,0,167,0,0,0,0,0,0


In [26]:
# Internal evidence for chain ids: bridges named after a chain, and where they go.
for b in ["arbitrum", "polygon", "gnosis", "omni", "megaeth", "superset"]:
    print(f"bridge {b!r}:", tf.loc[tf["bridge"] == b].groupby(["chain", "destinationChainId"]).size().to_dict())

bridge 'arbitrum': {('ethereum', 42161): 193}
bridge 'polygon': {('ethereum', 137): 14}
bridge 'gnosis': {('ethereum', 100): 6}
bridge 'omni': {('ethereum', 100): 8}
bridge 'megaeth': {('ethereum', 4326): 2}
bridge 'superset': {('arbitrum', 8453): 1}


In [27]:
# Same-chain rows: destination equals the source chain's id. The mapping below is an assumption for ethereum, optimism, bnb and base
# (no row in the data names those ids); arbitrum=42161, polygon=137 are supported by the chain-named bridges above.
SOURCE_ID_ASSUMED = {"ethereum": 1, "optimism": 10, "bnb": 56, "polygon": 137, "base": 8453, "arbitrum": 42161}
same_chain = tf["destinationChainId"] == tf["chain"].map(SOURCE_ID_ASSUMED)
print("rows whose destination id equals the assumed source-chain id:", int(same_chain.sum()))
tf[same_chain].groupby(["chain", "bridge"]).size().to_frame("rows")

rows whose destination id equals the assumed source-chain id:

 295


rows
chain    bridge              
arbitrum lifiIntentsDex    10
         m0Dex              2
         paxosDex           2
base     lifiIntentsDex   152
bnb      lifiIntentsDex    49
ethereum lifiIntentsDex    63
         m0Dex              5
         paxosDex           2
optimism lifiIntentsDex     6
polygon  lifiIntentsDex     4

In [28]:
# Destinations by source chain for the ten largest destination ids.
top_dest = dest.index[:10]
pd.crosstab(tf["chain"], tf["destinationChainId"].where(tf["destinationChainId"].isin(top_dest), other=-1).rename("destinationChainId (-1 = other)"))

destinationChainId (-1 = other),-1,1,10,56,137,999,4663,8453,42161,20000000000001,1151111081099710
chain,,,,,,,,,,,
arbitrum,1429,4362,524,2288,522,548,4571,3642,14,361,2137
base,4301,5559,1759,4573,1851,556,7601,152,4314,174,4068
bnb,2242,6848,244,49,2171,377,6309,3784,2320,148,2560
ethereum,3253,70,250,3849,1221,660,7582,3792,3291,754,4629
optimism,457,624,6,439,176,47,630,2266,924,14,173
polygon,607,1430,105,2136,4,80,343,1250,659,117,662


## 7. Native-asset rows

`sendingAssetId == 0x000…000` denotes the chain's native asset. `prices.csv` prices it under the same zero address on five chains (symbol `ETH` on arbitrum, base, ethereum, optimism; `BNB` on bnb) and has no row for polygon's native asset. No other sentinel (for example `0xeeee…`) occurs.

In [29]:
native = tf["sendingAssetId"] == ZERO_ADDRESS
print("native-asset rows:", int(native.sum()), f"({native.mean():.4%})")
print("rows with a 0xeeee… sentinel:", int(tf["sendingAssetId"].str.startswith("0xeeeeeeee").sum()))
display(price_tokens[price_tokens["token_address"] == ZERO_ADDRESS].sort_values("chain"))
nat = tf[native].groupby("chain").size().to_frame("native rows")
nat["share of chain"] = (nat["native rows"] / tf.groupby("chain").size()).round(4)
nat["priced in prices.csv"] = nat.index.isin(price_tokens.loc[price_tokens["token_address"] == ZERO_ADDRESS, "chain"])
nat

native-asset rows: 39940 (31.9883%)
rows with a 0xeeee… sentinel: 0


,chain,token_address,symbol,decimals
0,arbitrum,0x0000000000000000000000000000000000000000,ETH,18
5528,base,0x0000000000000000000000000000000000000000,ETH,18
14432,bnb,0x0000000000000000000000000000000000000000,BNB,18
18296,ethereum,0x0000000000000000000000000000000000000000,ETH,18
34706,optimism,0x0000000000000000000000000000000000000000,ETH,18


,native rows,share of chain,priced in prices.csv
chain,,,
arbitrum,6651,0.3261,True
base,13519,0.3873,True
bnb,3874,0.1432,True
ethereum,12797,0.4360,True
optimism,2735,0.4752,True
polygon,364,0.0492,False


In [30]:
# Polygon native rows cannot be priced or even decimal-adjusted from the data (no decimals row). Shown in raw units and, assuming 18 decimals, in whole tokens.
poly_native = tf[native & (tf["chain"] == "polygon")]
units_if_18 = poly_native["minAmount"].map(lambda a: float(Decimal(a) / Decimal(10) ** 18))
print("polygon native rows:", len(poly_native), " sum of minAmount:", sum(poly_native["minAmount"]), " whole tokens if 18 decimals:", round(units_if_18.sum(), 2))
print("bridges:", poly_native["bridge"].value_counts().to_dict())
print("top integrators:", poly_native["integrator"].value_counts().head(5).to_dict())

polygon native rows: 364  sum of minAmount: 300115588567767876364003  whole tokens if 18 decimals: 300115.59
bridges: {'layerswap': 340, 'near': 24}
top integrators: {'jumper.exchange': 132, 'rabbykmsv2': 95, 'lifi-api': 14, 'Zerion': 13, 'tangem': 8}


## 8. Duplicates

Three keys are checked: `transactionId` (LI.FI's own id), `(chain, tx_hash, log_index)` (the on-chain event identity), and `(chain, tx_hash)` (several events in one transaction).

In [31]:
dup_txid = tf.duplicated("transactionId", keep=False)
print("distinct transactionId:", tf["transactionId"].nunique(), " of", len(tf))
print("rows sharing a transactionId:", int(dup_txid.sum()), " distinct ids among them:", tf.loc[dup_txid, "transactionId"].nunique(), " max rows per id:", int(tf.groupby("transactionId").size().max()))
print("rows duplicated on (chain, tx_hash, log_index):", int(tf.duplicated(["chain", "tx_hash", "log_index"], keep=False).sum()))
print("fully identical rows:", int(tf.duplicated(keep=False).sum()))
print("tx_hash seen on more than one chain:", int((tf.groupby("tx_hash")["chain"].nunique() > 1).sum()))
cols = ["transactionId", "chain", "block_time", "tx_hash", "log_index", "bridge", "integrator", "sendingAssetId", "minAmount", "destinationChainId"]
tf[dup_txid].sort_values(["transactionId", "block_time"])[cols]

distinct transactionId: 124852  of 124858


rows sharing a transactionId: 12  distinct ids among them: 6  max rows per id: 2


rows duplicated on (chain, tx_hash, log_index): 0


fully identical rows: 0
tx_hash seen on more than one chain: 0


,transactionId,chain,block_time,tx_hash,log_index,bridge,integrator,sendingAssetId,minAmount,destinationChainId
3999,0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,optimism,2026-10-13 05:52:31+00:00,0x8388889759abc3ef731d501ee560ae82c31cdd9ce963c91e568a9e684b0bb002,287,relaydepository,epoch,0x94b008aa00579c1307b0ef2c499ad98a8ce58e58,98750,1
4115,0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,optimism,2026-10-13 06:00:29+00:00,0x07dfea310ddc4755e523ad42958381acd1738b70eb622652c37e299e1172df85,120,relaydepository,epoch,0x94b008aa00579c1307b0ef2c499ad98a8ce58e58,98750,1
8458,0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,bnb,2026-10-13 11:07:02+00:00,0x60f253000cc123b6ddf4985db78953994ceec39b6266a3fff61162fae780b4cb,557,relaydepository,sushi,0x5d3a1ff2b6bab83b63cd9ad0787074081a52ef34,49734893916554034264,324
8475,0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,bnb,2026-10-13 11:08:04+00:00,0xbbfe9f831e46648d049046162a547c4a229e718c159165d16fc611bbaa4883d1,311,relaydepository,sushi,0x5d3a1ff2b6bab83b63cd9ad0787074081a52ef34,49731962529614967125,324
25308,0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,arbitrum,2026-10-14 08:49:56+00:00,0x340cd5eacb476dad862c5156eb8dc78d6721e0bc5cd0a447fa10e8db17c17f86,5,across,jumper.exchange,0x0000000000000000000000000000000000000000,9998000000000000000,1
25310,0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,arbitrum,2026-10-14 08:50:14+00:00,0xddb372bd807764085dba892488487bb563e2567254a0dcf745f7ba25c7ed042b,5,across,jumper.exchange,0x0000000000000000000000000000000000000000,9998000000000000000,1
117561,0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,arbitrum,2026-10-19 14:50:21+00:00,0x5049d8d9f770ce76897ac019d1971ab34e3db215c0ce647cafedfabb81390beb,3,relaydepository,Barker,0x0000000000000000000000000000000000000000,997500000000000,999
117564,0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,arbitrum,2026-10-19 14:50:25+00:00,0x57a44a78a9c0ec9348d4f53c0b5eddbaff4876b22af291523fd7bb162a8ee90e,3,relaydepository,Barker,0x0000000000000000000000000000000000000000,997500000000000,999
122729,0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,ethereum,2026-10-19 20:41:23+00:00,0xbe72462d374d4d1f89c64fb4f97ee6fa4b067711de7c775ff9a21a89176ec865,336,gasZipBridge,qern,0x0000000000000000000000000000000000000000,9935000000000,8453
122799,0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,ethereum,2026-10-19 20:46:11+00:00,0x8cac270c2ecea6b97850417748cb828ad9c4d831f4d0827f47e244cf0acef88b,299,gasZipBridge,qern,0x0000000000000000000000000000000000000000,9935000000000,8453


In [32]:
# The six transactionId pairs: same chain, bridge, integrator, token and destination, different tx_hash, seconds to minutes apart; five of six with the same amount.
pairs = tf[dup_txid].groupby("transactionId").agg(rows=("tx_hash", "size"), tx_hashes=("tx_hash", "nunique"), chains=("chain", "nunique"), amounts=("minAmount", "nunique"),
                                                 bridges=("bridge", "nunique"), integrators=("integrator", "nunique"), seconds_apart=("block_time", lambda s: (s.max() - s.min()).total_seconds()))
pairs

,rows,tx_hashes,chains,amounts,bridges,integrators,seconds_apart
transactionId,,,,,,,
0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,2,2,1,1,1,1,478.0
0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,2,2,1,2,1,1,62.0
0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,2,2,1,1,1,1,18.0
0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,2,2,1,1,1,1,4.0
0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,2,2,1,1,1,1,288.0
0xee199f180cc6ca794e94e368ccd676ad714bb39406b07ec97966948787af9b02,2,2,1,1,1,1,7.0


In [33]:
# Several LiFiTransferStarted events in one transaction: each has its own transactionId.
events_per_tx = tf.groupby(["chain", "tx_hash"]).size()
print("events per (chain, tx_hash):", events_per_tx.value_counts().sort_index().to_dict())
multi = tf[tf.duplicated(["chain", "tx_hash"], keep=False)].sort_values(["chain", "tx_hash", "log_index"])
print("rows in multi-event transactions:", len(multi), " distinct transactionIds per such tx:", multi.groupby(["chain", "tx_hash"])["transactionId"].nunique().unique().tolist())
multi[cols]

events per (chain, tx_hash): {1: 124836, 2: 11}
rows in multi-event transactions: 22  distinct transactionIds per such tx: [2]


,transactionId,chain,block_time,tx_hash,log_index,bridge,integrator,sendingAssetId,minAmount,destinationChainId
29121,0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e,base,2026-10-14 13:03:39+00:00,0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d,247,gasZipBridge,base-app,0x0000000000000000000000000000000000000000,360013500000000,56
29122,0xf4631bab03ec9937436087296d8c6ca6960e388155536d00e1417c4113fa9bdd,base,2026-10-14 13:03:39+00:00,0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d,271,mayan,base-app,0x0000000000000000000000000000000000000000,38810828038075959,1151111081099710
19161,0xfc35543ffa183de35be95808bc1f6c9ab9069bff0d9ff8982c7c15edfcc74c79,base,2026-10-13 23:28:07+00:00,0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9,785,near,base-app,0x833589fcd6edb6e08f4c7c32d4f71b54bda02913,5543223,1151111081099710
19162,0x6a9be92dc99bf108a4f04b5184b49e76e6d22dbd84c362f223cd31d5394a54f8,base,2026-10-13 23:28:07+00:00,0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9,839,mayan,base-app,0x833589fcd6edb6e08f4c7c32d4f71b54bda02913,15062258,56
51595,0x55c37f48b62959b47fde6bec564f28c762ac7f63f0bf64fd4bddb826abdef12f,base,2026-10-15 15:35:59+00:00,0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747,509,mayan,base-app,0x0000000000000000000000000000000000000000,98885452357043226,1151111081099710
51596,0x7a3678e4f384d544d13720d843e6ac98cad81dd23726a518aa87179e430f0e05,base,2026-10-15 15:35:59+00:00,0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747,536,gasZipBridge,base-app,0x0000000000000000000000000000000000000000,724494232449239,42161
117108,0x0b54fa2c6a21175c73e0338966e7d08e17e8d0482b8767a6fcd779a2f1fd2797,base,2026-10-19 14:21:45+00:00,0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982,546,mayan,base-app,0x833589fcd6edb6e08f4c7c32d4f71b54bda02913,15517128,137
117109,0x2e685da79dc48d704cc86a1014b1aa984c6bd2aa4bb57ad1bd891eae908ac0af,base,2026-10-19 14:21:45+00:00,0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982,570,mayan,base-app,0x0000000000000000000000000000000000000000,3926422413811648,1151111081099710
100308,0x6c939c84de84c74704e04d7c4882695687245309c36d4d2578dbeed44b64789b,base,2026-10-18 14:33:47+00:00,0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95,1412,mayan,base-app,0x0000000000000000000000000000000000000000,4399427419527825,1151111081099710
100309,0xa2eeb4dae124f0091c209649c37d15ede479521fda639f701ee89bf0659e8461,base,2026-10-18 14:33:47+00:00,0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95,1438,mayan,base-app,0x0000000000000000000000000000000000000000,2274591806664041,1151111081099710


## 9. Referrer usage

`referrer` is an address passed with the call; the zero address means none.

In [34]:
ref_zero = tf["referrer"] == ZERO_ADDRESS
print("rows with zero referrer:", int(ref_zero.sum()), f"({ref_zero.mean():.4%})", "  non-zero:", int((~ref_zero).sum()), "  distinct non-zero referrers:", tf.loc[~ref_zero, "referrer"].nunique())
nz = tf[~ref_zero]
print("non-zero referrer equals tx_from:", int((nz["referrer"] == nz["tx_from"]).sum()), "  equals receiver:", int((nz["referrer"] == nz["receiver"]).sum()))
display(nz["referrer"].value_counts().to_frame("rows"))
nz.groupby("integrator")["referrer"].agg(rows="size", distinct_referrers="nunique").sort_values("rows", ascending=False)

rows with zero referrer: 122796 (98.3485%)   non-zero: 2062   distinct non-zero referrers: 21
non-zero referrer equals tx_from: 2   equals receiver: 2


,rows
referrer,
0xb4f34d09124b8c9712957b76707b42510041ecbb,969
0xfaae4ef139ace211845db13b4c9c13169cc2a431,827
0x32303ffcb9b6564c2b8a373433a043a7f17e4b37,61
0x190985cfcaa72062781a0558c1dd0ad7c1fb3986,44
0xb8af0fa3e38e8cb95870091b0d4e32ca232b780d,43
0x7225b33c16fba2dcb38b0acbcc562a72ef576a6b,40
0xee90af848149794796e4cba08472957b33b77b05,12
0x19a1e4639a37e83ac888727e1bfe8a2970906ee9,11
0x2e6168f9ca3fe204a2110c4613ce18985f3fbf39,10


,rows,distinct_referrers
integrator,,
safepal,969,1
HashPort-Wallet-Prod,827,1
debox,120,4
waycard-web,44,1
layer3,40,1
coinfello-web,16,3
wigwam,12,1
dain-trader-crosschain,10,1
HashPort-Wallet-Dev,7,1


## 10. The two boolean flags

`hasSourceSwaps`: a swap ran on the source chain before bridging, so `sendingAssetId`/`minAmount` describe what was bridged, not what the user paid in. `hasDestinationCall`: a contract call runs on the destination after bridging.

In [35]:
flags = tf.groupby(["hasSourceSwaps", "hasDestinationCall"]).size().to_frame("rows")
flags["share"] = (flags["rows"] / len(tf)).round(4)
print("hasSourceSwaps true:", int(tf["hasSourceSwaps"].sum()), f"({tf['hasSourceSwaps'].mean():.4%})", "  hasDestinationCall true:", int(tf["hasDestinationCall"].sum()), f"({tf['hasDestinationCall'].mean():.4%})")
display(flags)
display(tf.groupby("chain")[["hasSourceSwaps", "hasDestinationCall"]].mean().round(4))
display(tf.groupby("bridge")[["hasSourceSwaps", "hasDestinationCall"]].agg(["mean", "size"]).round(4).sort_values(("hasDestinationCall", "mean"), ascending=False).head(12))
tf.groupby("integrator")[["hasSourceSwaps", "hasDestinationCall"]].agg(["mean", "size"]).round(4).sort_values(("hasSourceSwaps", "size"), ascending=False).head(12)

hasSourceSwaps true: 84679 (67.8202%)   hasDestinationCall true: 3955 (3.1676%)


rows   share
hasSourceSwaps hasDestinationCall               
False          False               39717  0.3181
               True                  462  0.0037
True           False               81186  0.6502
               True                 3493  0.0280

,hasSourceSwaps,hasDestinationCall
chain,,
arbitrum,0.5397,0.0196
base,0.7340,0.0291
bnb,0.6911,0.0166
ethereum,0.6768,0.0615
optimism,0.7087,0.0179
polygon,0.7315,0.0248


hasSourceSwaps        hasDestinationCall       
                          mean   size               mean   size
bridge                                                         
stargateV2              0.8320    488             0.1988    488
across                  0.7002  34163             0.1129  34163
mayanMCTP               0.7691    433             0.0000    433
superset                1.0000      1             0.0000      1
squid                   0.9939    494             0.0000    494
relaydepository         0.7013  31483             0.0000  31483
polymerStandard         0.2984   6631             0.0000   6631
polymer                 0.5189   1320             0.0000   1320
polygon                 0.8571     14             0.0000     14
paxosDex                1.0000      4             0.0000      4
paxos                   0.3182     22             0.0000     22
omni                    0.2500      8             0.0000      8

hasSourceSwaps        hasDestinationCall       
                          mean   size               mean   size
integrator                                                     
jumper.exchange         0.5103  40908             0.0139  40908
_binancewallet          0.3787  15109             0.0201  15109
rabbykmsv2              0.9999   7759             0.0000   7759
metamask-bridge         0.9347   7332             0.0953   7332
base-app                0.9840   5859             0.0340   5859
phantom                 0.9994   5233             0.2650   5233
lifi-api                0.9997   3465             0.0124   3465
tangem                  1.0000   2621             0.0145   2621
bitget                  1.0000   2534             0.0000   2534
Cow                     0.0724   2237             0.0000   2237
amahub                  0.9995   1875             0.0037   1875
_arbitrum               0.0261   1688             0.0118   1688

In [36]:
# Over all 598 integrator strings, not only the twelve with the most rows: how many set each flag on every row, on no row, or on some rows.
per_integrator = tf.groupby("integrator").agg(rows=("hasSourceSwaps", "size"), source_swaps=("hasSourceSwaps", "mean"), destination_call=("hasDestinationCall", "mean"))
def always_never(col):
    v = per_integrator[col]
    return {"always (100%)": int((v == 1).sum()), "never (0%)": int((v == 0).sum()), "mixed": int(((v > 0) & (v < 1)).sum()),
            "rows under 'always'": int(per_integrator.loc[v == 1, "rows"].sum()), "rows under 'never'": int(per_integrator.loc[v == 0, "rows"].sum())}
print("integrator strings:", len(per_integrator))
display(pd.DataFrame({c: always_never(c) for c in ["source_swaps", "destination_call"]}))
# The mixed integrators with at least 100 rows, lowest source-swap share first.
per_integrator[(per_integrator["rows"] >= 100) & (per_integrator["source_swaps"] > 0) & (per_integrator["source_swaps"] < 1)].sort_values("source_swaps").round(4)

integrator strings: 598


,source_swaps,destination_call
always (100%),535,10
never (0%),14,505
mixed,49,83
rows under 'always',23878,271
rows under 'never',328,23816


,rows,source_swaps,destination_call
integrator,,,
derive-v3,535,0.0019,0.0000
_arbitrum,1688,0.0261,0.0118
oku.trade,338,0.0562,0.0089
Cow,2237,0.0724,0.0000
jumperadvanced,1119,0.0760,0.0036
os-prod,1540,0.1110,0.0675
prjx,146,0.1301,0.0068
backpack-wallet,373,0.1501,0.0054
HashPort-Wallet-Prod,827,0.1596,0.0012


In [37]:
# Sender and receiver: how often the user receives at their own address.
print("receiver == tx_from:", int((tf['receiver'] == tf['tx_from']).sum()), f"({(tf['receiver'] == tf['tx_from']).mean():.4%})")
print("distinct tx_from:", tf["tx_from"].nunique(), "  distinct receiver:", tf["receiver"].nunique())
tf["tx_from"].value_counts().head(5).to_frame("rows")

receiver == tx_from: 84642 (67.7906%)


distinct tx_from: 56280   distinct receiver: 58889


,rows
tx_from,
0x7202932b3be70edf0657d5bada261d610e0d7db9,1685
0xe469ee44d92bb173b7edc576ebb09da55dbfe32e,986
0x1278c1e48e3c9548a5d9f2b16dc27ed311b0697c,974
0x54e2acab04c89a3fe02852bf8dd69ee8f526bc75,971
0xca1ef33a9522ef11265858592f0209b45473c846,806


## 11. A first USD amount column

`amount_token = minAmount / 10**decimals` with exact integer arithmetic (`Decimal`), then `amount_usd = amount_token × price_usd` for the row's `(chain, sendingAssetId, hour)`. This is the simplest candidate definition, not the one in `task_1.md`: it uses `minAmount` (the source-side amount handed to the bridge, not the amount received), the hour's average price, and no fallback when the price is missing. Rows without a price get `NaN` and are counted, not imputed.

In [38]:
def token_amount(min_amount, decimals):
    if pd.isna(decimals):
        return float("nan")
    return float(Decimal(min_amount) / Decimal(10) ** int(decimals))

joined["amount_token"] = [token_amount(a, d) for a, d in zip(joined["minAmount"], joined["decimals"])]
joined["amount_usd"] = joined["amount_token"] * joined["price_usd"]
unpriced = joined["amount_usd"].isna()
print("rows priced:", int((~unpriced).sum()), "  rows not priced:", int(unpriced.sum()), f"({unpriced.mean():.4%})")
print("total amount_usd over priced rows:", round(joined["amount_usd"].sum(), 2))
joined["amount_usd"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_frame("amount_usd")

rows priced: 124477   rows not priced: 381 (0.3051%)
total amount_usd over priced rows: 307935992.59


,amount_usd
count,1.244770e+05
mean,2.473838e+03
std,6.884798e+04
min,3.500725e-05
50%,4.353573e+01
90%,1.622969e+03
99%,3.000452e+04
99.9%,3.000378e+05
max,1.000242e+07


In [39]:
usd_by_chain = joined.groupby("chain")["amount_usd"].agg(usd="sum", priced_rows="count", median_usd="median")
usd_by_chain["unpriced_rows"] = joined.groupby("chain")["amount_usd"].apply(lambda s: int(s.isna().sum()))
usd_by_chain["usd_share"] = (usd_by_chain["usd"] / usd_by_chain["usd"].sum()).round(4)
usd_by_chain.sort_values("usd", ascending=False).round(2)

,usd,priced_rows,median_usd,unpriced_rows,usd_share
chain,,,,,
ethereum,1.636540e+08,29349,76.13,2,0.53
arbitrum,9.021959e+07,20394,104.45,4,0.29
base,2.621541e+07,34906,24.17,2,0.09
bnb,1.606487e+07,27048,27.50,4,0.05
polygon,7.634459e+06,7027,36.69,366,0.02
optimism,4.147697e+06,5753,17.63,3,0.01


In [40]:
display(joined.groupby("bridge")["amount_usd"].agg(usd="sum", rows="size").assign(usd_share=lambda d: (d["usd"] / d["usd"].sum()).round(4)).sort_values("usd", ascending=False).head(12).round(2))
joined.groupby("integrator")["amount_usd"].agg(usd="sum", rows="size").assign(usd_share=lambda d: (d["usd"] / d["usd"].sum()).round(4)).sort_values("usd", ascending=False).head(15).round(2)

,usd,rows,usd_share
bridge,,,
polymerStandard,1.297023e+08,6631,0.42
lifiIntents,4.016717e+07,16185,0.13
relaydepository,3.402573e+07,31483,0.11
glacis,2.804719e+07,633,0.09
across,2.471787e+07,34163,0.08
mayan,1.919399e+07,11422,0.06
near,1.136946e+07,2814,0.04
layerswap,7.577968e+06,12659,0.02
celercircle,4.413173e+06,258,0.01


,usd,rows,usd_share
integrator,,,
jumper.exchange,1.087273e+08,40908,0.35
_arbitrum,6.159366e+07,1688,0.20
Cow,2.530773e+07,2237,0.08
_binancewallet,2.494595e+07,15109,0.08
Pharos-Bridge,1.679054e+07,40,0.05
jumperadvanced,9.961745e+06,1119,0.03
USDai,9.656884e+06,62,0.03
HashPort-Wallet-Prod,5.693478e+06,827,0.02
rabbykmsv2,5.651888e+06,7759,0.02


In [41]:
# Concentration: a few very large stablecoin transfers dominate the total.
total_usd = joined["amount_usd"].sum()
for cut in (1e5, 1e6, 1e7):
    big = joined["amount_usd"] > cut
    print(f"rows above ${cut:,.0f}: {int(big.sum()):5d}   their USD: {joined.loc[big, 'amount_usd'].sum():,.0f}   share of total: {joined.loc[big, 'amount_usd'].sum() / total_usd:.4f}")
print("rows below $1:", int((joined["amount_usd"] < 1).sum()), "  below $0.01:", int((joined["amount_usd"] < 0.01).sum()))
joined.sort_values("amount_usd", ascending=False)[["chain", "block_time", "bridge", "integrator", "symbol", "decimals", "minAmount", "amount_token", "price_usd", "amount_usd", "destinationChainId", "hasSourceSwaps"]].head(20)

rows above $100,000:   382   their USD: 168,497,252   share of total: 0.5472
rows above $1,000,000:    22   their USD: 76,115,424   share of total: 0.2472
rows above $10,000,000:     3   their USD: 30,005,413   share of total: 0.0974
rows below $1: 11422   below $0.01: 53


,chain,block_time,bridge,integrator,symbol,decimals,minAmount,amount_token,price_usd,amount_usd,destinationChainId,hasSourceSwaps
102200,ethereum,2026-10-18 16:40:35+00:00,polymerStandard,_arbitrum,USDC,6.0,10000000000000,1.000000e+07,1.000242,1.000242e+07,42161,False
101908,ethereum,2026-10-18 16:19:23+00:00,polymerStandard,_arbitrum,USDC,6.0,10000000000000,1.000000e+07,1.000242,1.000242e+07,42161,False
67750,ethereum,2026-10-16 13:39:59+00:00,polymerStandard,Pharos-Bridge,USDC,6.0,10000000000000,1.000000e+07,1.000057,1.000057e+07,1672,False
102526,ethereum,2026-10-18 17:01:59+00:00,polymerStandard,_arbitrum,USDC,6.0,9519447993443,9.519448e+06,1.000216,9.521501e+06,42161,False
120301,arbitrum,2026-10-19 17:57:04+00:00,polymerStandard,_arbitrum,USDC,6.0,9519447993443,9.519448e+06,1.000149,9.520870e+06,1,False
67995,ethereum,2026-10-16 13:54:59+00:00,polymerStandard,Pharos-Bridge,USDC,6.0,6504986300000,6.504986e+06,1.000057,6.505357e+06,1672,False
38466,arbitrum,2026-10-15 00:04:58+00:00,glacis,jumper.exchange,PYUSD,6.0,2011386122272,2.011386e+06,1.000957,2.013311e+06,1,False
19642,ethereum,2026-10-14 00:23:47+00:00,polymerStandard,_binancewallet,USDC,6.0,2000000001600,2.000000e+06,1.000189,2.000378e+06,1672,False
38956,arbitrum,2026-10-15 00:53:01+00:00,glacis,USDai,PYUSD,6.0,1653423864597,1.653424e+06,1.000957,1.655006e+06,1,True
70021,ethereum,2026-10-16 15:52:23+00:00,glacis,safe-app,USDG,6.0,1517016080887,1.517016e+06,1.000488,1.517757e+06,4663,True


In [42]:
by_symbol = joined.groupby("symbol")["amount_usd"].agg(usd="sum", rows="size").sort_values("usd", ascending=False)
by_symbol["usd_share"] = (by_symbol["usd"] / total_usd).round(4)
print("USD share of symbols containing 'USD':", round(joined.loc[joined["symbol"].fillna("").str.contains("USD"), "amount_usd"].sum() / total_usd, 4))
by_symbol.head(15).round(2)

USD share of symbols containing 'USD': 0.8762


,usd,rows,usd_share
symbol,,,
USDC,2.123609e+08,48285,0.69
ETH,3.111770e+07,41135,0.10
USDT,2.633215e+07,17683,0.09
USDG,1.398172e+07,466,0.05
PYUSD,1.175533e+07,56,0.04
WETH,3.937325e+06,5294,0.01
USD₮0,3.406606e+06,2288,0.01
DAI,1.216423e+06,236,0.00
USDT0,1.056403e+06,2649,0.00


In [43]:
# How the oddities of sections 6, 8 and 10 look in USD under this first definition.
joined["same_chain"] = same_chain.values
joined["dup_transaction_id"] = dup_txid.values
joined["multi_event_tx"] = tf.duplicated(["chain", "tx_hash"], keep=False).values
joined["dest_above_2_32"] = above32.values
joined["underscore_integrator"] = tf["integrator"].str.startswith("_").values
effects = {}
for name in ["same_chain", "dup_transaction_id", "multi_event_tx", "dest_above_2_32", "underscore_integrator", "hasSourceSwaps", "hasDestinationCall"]:
    m = joined[name].astype(bool)
    effects[name] = {"rows": int(m.sum()), "usd": round(float(joined.loc[m, "amount_usd"].sum()), 2), "usd_share": round(float(joined.loc[m, "amount_usd"].sum() / total_usd), 4)}
pd.DataFrame(effects).T

,rows,usd,usd_share
same_chain,295.0,226443.54,0.0007
dup_transaction_id,12.0,50357.82,0.0002
multi_event_tx,22.0,7744.05,0.0000
dest_above_2_32,16666.0,31878832.30,0.1035
underscore_integrator,19068.0,88738885.13,0.2882
hasSourceSwaps,84679.0,64501904.76,0.2095
hasDestinationCall,3955.0,2114370.04,0.0069


## 12. Oddity sizes in one table

Every oddity the profile found, recomputed here in one place with rows, share of rows, and USD under the §11 column where it applies.

In [44]:
n = len(tf)
oddities = [
    ("minAmount exceeds int64 (must stay exact integer)", int((tf["minAmount"] > 2**63 - 1).sum()), None, "§1"),
    ("integrator is the empty string", int((tf["integrator"] == "").sum()), None, "§4"),
    ("integrator has a leading underscore", int(tf["integrator"].str.startswith("_").sum()), effects["underscore_integrator"]["usd"], "§4"),
    ("integrator in a multi-name family (env suffixes)", len(fam_rows), None, "§4"),
    ("token never priced in prices.csv", int((joined["price_missing"] & ~joined["token_priced_any_hour"]).sum()), None, "§5"),
    ("  of which polygon native asset", len(poly_native), None, "§5, §7"),
    ("token priced but not in this hour", int((joined["price_missing"] & joined["token_priced_any_hour"]).sum()), None, "§5"),
    ("destination id above 2^32-1", int(above32.sum()), effects["dest_above_2_32"]["usd"], "§6"),
    ("receiver is the 0x11f1…f1 sentinel", int(sentinel.sum()), None, "§6"),
    ("destination equals assumed source-chain id", int(same_chain.sum()), effects["same_chain"]["usd"], "§6"),
    ("native-asset rows (zero address)", int(native.sum()), round(float(joined.loc[native.values, "amount_usd"].sum()), 2), "§7"),
    ("rows sharing a transactionId", int(dup_txid.sum()), effects["dup_transaction_id"]["usd"], "§8"),
    ("rows in a transaction with two events", int(joined["multi_event_tx"].sum()), effects["multi_event_tx"]["usd"], "§8"),
    ("non-zero referrer", int((~ref_zero).sum()), None, "§9"),
    ("hasSourceSwaps true", int(tf["hasSourceSwaps"].sum()), effects["hasSourceSwaps"]["usd"], "§10"),
    ("hasDestinationCall true", int(tf["hasDestinationCall"].sum()), effects["hasDestinationCall"]["usd"], "§10"),
    ("rows not priced under the first USD column", int(unpriced.sum()), None, "§11"),
    ("rows above $1M", int((joined["amount_usd"] > 1e6).sum()), round(float(joined.loc[joined["amount_usd"] > 1e6, "amount_usd"].sum()), 2), "§11"),
]
summary = pd.DataFrame(oddities, columns=["oddity", "rows", "usd (first column)", "section"])
summary["share of rows"] = (summary["rows"] / n).round(4)
summary[["oddity", "rows", "share of rows", "usd (first column)", "section"]]

,oddity,rows,share of rows,usd (first column),section
0,minAmount exceeds int64 (must stay exact integer),14864,0.1190,NaN,§1
1,integrator is the empty string,2,0.0000,NaN,§4
2,integrator has a leading underscore,19068,0.1527,88738885.13,§4
3,integrator in a multi-name family (env suffixes),11252,0.0901,NaN,§4
4,token never priced in prices.csv,380,0.0030,NaN,§5
5,of which polygon native asset,364,0.0029,NaN,"§5, §7"
6,token priced but not in this hour,1,0.0000,NaN,§5
7,destination id above 2^32-1,16666,0.1335,31878832.30,§6
8,receiver is the 0x11f1…f1 sentinel,16668,0.1335,NaN,§6
9,destination equals assumed source-chain id,295,0.0024,226443.54,§6
